# Experimento 4: comparacion de resampleo a 50 Hz y 20 Hz

**Objetivo:** evaluar si la unificacion de UPFall, KFall y UMAFall a 50 Hz o a 20 Hz conserva una representacion util de las senales inerciales.

Este experimento analiza exclusivamente la fidelidad del resampleo. No entrena modelos ni modifica la capa oro. Los datos se leen desde `bronce/falls` en Azure Data Lake y todos los resultados se muestran inline.

## Hipotesis y alcance

Se comparan dos lineas independientes:

- **Linea 1:** UPFall/KFall `100 -> 50 Hz` y UMAFall `20 -> 50 Hz`.
- **Linea 2:** UPFall/KFall `100 -> 20 Hz` y UMAFall `20 -> 20 Hz` sin modificar.

Se procesan todos los trials `Fall` y `ADL`. Las visualizaciones usan casos P05, P50 y P95 seleccionados por error absoluto del pico de `AVM`. Los umbrales de referencia son Pearson `>= 0.85`, desfase `<= 100 ms` y error de pico `<= 25%`.

**Advertencia:** el upsampling `20 -> 50 Hz` interpola muestras; no recupera informacion fisica por encima del Nyquist original de UMAFall (`10 Hz`).

In [ ]:
%pip install numpy pandas scipy dtaidistance matplotlib azure-storage-file-datalake --quiet

In [ ]:
import io
from math import gcd

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dtaidistance import dtw as dtw_lib
from google.colab import userdata
from IPython.display import display
from scipy.signal import butter, filtfilt, resample_poly, welch
from scipy.stats import pearsonr
from azure.storage.filedatalake import DataLakeServiceClient

plt.rcParams.update({
    'figure.dpi': 110,
    'axes.grid': True,
    'grid.linestyle': '--',
    'grid.alpha': 0.35,
})

DATASETS_META = {
    'UPFall': {'fs': 100, 'csv': 'UPFall-Reduced.csv'},
    'KFall': {'fs': 100, 'csv': 'KFall-Reduced.csv'},
    'UMAFall': {'fs': 20, 'csv': 'UMAFall-Reduced.csv'},
}
TARGETS = {
    'Linea 1 - 50 Hz': 50,
    'Linea 2 - 20 Hz': 20,
}
VALID_LABELS = {'Fall', 'ADL'}
SENSORS = ('AVM', 'GVM')
METRICS = ('snr_db', 'pearson_r', 'dtw_norm', 'phase_shift_ms', 'peak_atten_pct')
EVAL_WINDOW_SEC = 2.0
KAISER_BETA = 5.0
THR_PEARSON_MIN = 0.85
THR_PHASE_MS_MAX = 100.0
THR_ATTEN_PCT_MAX = 25.0
QUANTILES = ((0.05, 'P05'), (0.50, 'P50'), (0.95, 'P95'))
RAW_SCHEMA_COLS = [
    'Subject', 'Activity_Label', 'Activity_Code', 'Trial', 'Sample_Index',
    'Ax', 'Ay', 'Az', 'Gx', 'Gy', 'Gz',
]

print('Configuracion cargada:')
print(f'  Datasets: {list(DATASETS_META)}')
print(f'  Objetivos: {list(TARGETS.values())} Hz')
print(f'  Ventana de evaluacion: {EVAL_WINDOW_SEC} s')

In [ ]:
# Carga desde Azure Data Lake: mismo patron usado por los notebooks existentes.
CONNECTION_STRING = userdata.get('cadenaAzure')
service_client = DataLakeServiceClient.from_connection_string(CONNECTION_STRING)
fs_bronce = service_client.get_file_system_client('bronce')
dir_falls = fs_bronce.get_directory_client('falls')

def load_csv_from_bronze(filename):
    content = dir_falls.get_file_client(filename).download_file().readall()
    return pd.read_csv(io.BytesIO(content))

raw_datasets = {}
for dataset_name, meta in DATASETS_META.items():
    df = load_csv_from_bronze(meta['csv'])
    missing = sorted(set(RAW_SCHEMA_COLS) - set(df.columns))
    if missing:
        raise ValueError(f'[{dataset_name}] faltan columnas: {missing}')
    unknown_labels = set(df['Activity_Label'].dropna().astype(str)) - VALID_LABELS
    if unknown_labels:
        raise ValueError(f'[{dataset_name}] etiquetas inesperadas: {unknown_labels}')
    raw_datasets[dataset_name] = df
    n_trials = df[['Subject', 'Activity_Code', 'Trial']].drop_duplicates().shape[0]
    print(f'  {dataset_name:8s} | {meta["fs"]:3d} Hz | {len(df):>10,} filas | {n_trials:>5,} trials')

assert set(raw_datasets) == set(DATASETS_META)
print('Carga completa.')

## Funciones de resampleo y medicion

El resampleo se aplica sobre el trial completo. Para downsampling, la referencia original se limita a la banda representable por la frecuencia objetivo. Para upsampling y para `20 -> 20`, la referencia conserva la senal original.

In [ ]:
def get_poly_factors(fs_orig, fs_target):
    common = gcd(int(fs_orig), int(fs_target))
    return int(fs_target) // common, int(fs_orig) // common


def resample_signal(signal, fs_orig, fs_target, kaiser_beta=KAISER_BETA):
    signal = np.asarray(signal, dtype=float)
    if fs_orig == fs_target:
        return signal.copy()
    up, down = get_poly_factors(fs_orig, fs_target)
    return resample_poly(signal, up=up, down=down, window=('kaiser', kaiser_beta))


def reference_in_target_band(signal, fs_orig, fs_target):
    signal = np.asarray(signal, dtype=float)
    if fs_orig <= fs_target:
        return signal.copy()
    cutoff = fs_target / 2.0 - 0.5
    if cutoff <= 0 or len(signal) < 32:
        return signal.copy()
    b, a = butter(8, cutoff / (fs_orig / 2.0), btype='low')
    return filtfilt(b, a, signal)


def calculate_magnitudes(trial_df):
    trial_df = trial_df.sort_values('Sample_Index')
    acc = trial_df[['Ax', 'Ay', 'Az']].astype(float).to_numpy()
    gyro = trial_df[['Gx', 'Gy', 'Gz']].astype(float).to_numpy()
    return np.linalg.norm(acc, axis=1), np.linalg.norm(gyro, axis=1)


def aligned_window(signal, fs, center_time, window_sec=EVAL_WINDOW_SEC):
    half_window = window_sec / 2.0
    start = max(0, int(round((center_time - half_window) * fs)))
    stop = min(len(signal), int(round((center_time + half_window) * fs)))
    times = np.arange(start, stop) / fs - center_time
    return np.asarray(signal[start:stop], dtype=float), times


def interpolate_resampled_to_original(orig_times, resampled, res_times):
    return np.interp(orig_times, res_times, resampled)


def safe_pearson(orig, other):
    if np.std(orig) == 0 or np.std(other) == 0:
        return 1.0 if np.allclose(orig, other) else 0.0
    return float(pearsonr(orig, other)[0])


def zscore(signal):
    scale = np.std(signal)
    return (signal - np.mean(signal)) / scale if scale > 0 else signal - np.mean(signal)


def normalized_dtw(orig, resampled):
    a = zscore(orig).astype(np.double)
    b = zscore(resampled).astype(np.double)
    max_len = 500
    if len(a) > max_len:
        a = a[:max_len]
        b = b[:max_len]
    if len(a) < 2 or len(b) < 2:
        return np.nan
    distance = dtw_lib.distance_fast(a, b)
    return float(distance / max(len(a), len(b)))

In [ ]:
def calculate_window_metrics(orig_raw, orig_reference, resampled, orig_times, res_times, fs_orig, fs_target):
    resampled_on_orig = interpolate_resampled_to_original(orig_times, resampled, res_times)
    noise = orig_reference - resampled_on_orig
    signal_power = np.mean(orig_reference ** 2)
    noise_power = np.mean(noise ** 2)
    snr_db = np.inf if noise_power <= np.finfo(float).eps else 10 * np.log10(signal_power / noise_power)
    max_reference = np.max(orig_reference)
    max_resampled = np.max(resampled_on_orig)
    peak_error = (abs(max_reference - max_resampled) / abs(max_reference) * 100.0
                  if abs(max_reference) > np.finfo(float).eps else 0.0)
    return {
        'snr_db': float(snr_db),
        'pearson_r': safe_pearson(orig_reference, resampled_on_orig),
        'dtw_norm': normalized_dtw(orig_reference, resampled),
        'peak_atten_pct': float(peak_error),
    }


def analyze_trial(trial_df, fs_orig, fs_target):
    trial_df = trial_df.sort_values('Sample_Index')
    if len(trial_df) < 2:
        raise ValueError('Trial sin muestras suficientes para calcular metricas')
    avm, gvm = calculate_magnitudes(trial_df)
    raw_signals = {'AVM': avm, 'GVM': gvm}
    resampled_signals = {
        sensor: resample_signal(signal, fs_orig, fs_target)
        for sensor, signal in raw_signals.items()
    }
    reference_signals = {
        sensor: reference_in_target_band(signal, fs_orig, fs_target)
        for sensor, signal in raw_signals.items()
    }
    center_time = int(np.argmax(reference_signals['AVM'])) / fs_orig
    windows = {}
    metrics = {}
    for sensor in SENSORS:
        raw_window, orig_times = aligned_window(raw_signals[sensor], fs_orig, center_time)
        reference_window, _ = aligned_window(reference_signals[sensor], fs_orig, center_time)
        resampled_window, res_times = aligned_window(resampled_signals[sensor], fs_target, center_time)
        if min(len(raw_window), len(reference_window), len(resampled_window)) < 2:
            raise ValueError('Ventana sin muestras suficientes para calcular metricas')
        sensor_metrics = calculate_window_metrics(
            raw_window, reference_window, resampled_window,
            orig_times, res_times, fs_orig, fs_target,
        )
        raw_peak_time = int(np.argmax(reference_signals[sensor])) / fs_orig
        resampled_peak_time = int(np.argmax(resampled_signals[sensor])) / fs_target
        sensor_metrics['phase_shift_ms'] = abs(raw_peak_time - resampled_peak_time) * 1000.0
        metrics[sensor] = sensor_metrics
        windows[sensor] = {
            'raw': raw_window,
            'reference': reference_window,
            'resampled': resampled_window,
            'orig_times': orig_times,
            'res_times': res_times,
        }
    return {
        'raw_signals': raw_signals,
        'reference_signals': reference_signals,
        'resampled_signals': resampled_signals,
        'windows': windows,
        'metrics': metrics,
        'center_time': center_time,
    }


def passes_thresholds(row):
    return (
        row['pearson_r'] >= THR_PEARSON_MIN
        and row['phase_shift_ms'] <= THR_PHASE_MS_MAX
        and row['peak_atten_pct'] <= THR_ATTEN_PCT_MAX
    )


def run_trial_metrics(df, dataset_name, fs_orig, fs_target, line_name):
    rows = []
    trial_cols = ['Subject', 'Activity_Code', 'Trial']
    for keys, trial_df in df.groupby(trial_cols, sort=True):
        if trial_df['Activity_Label'].nunique() != 1:
            raise ValueError(f'{dataset_name} trial mixto: {keys}')
        analysis = analyze_trial(trial_df, fs_orig, fs_target)
        if analysis is None:
            raise ValueError(f'{dataset_name} trial omitido: {keys}')
        label = str(trial_df['Activity_Label'].iloc[0])
        for sensor in SENSORS:
            row = {
                'line': line_name,
                'Dataset': dataset_name,
                'Activity_Label': label,
                'Subject': keys[0],
                'Activity_Code': keys[1],
                'Trial': keys[2],
                'sensor': sensor,
                'fs_orig': fs_orig,
                'fs_target': fs_target,
                'sample_ratio': fs_target / fs_orig,
                'interpolated_fraction': max(0.0, 1.0 - fs_orig / fs_target),
            }
            row.update(analysis['metrics'][sensor])
            row['quality_pass'] = passes_thresholds(row)
            rows.append(row)
    frame = pd.DataFrame(rows)
    invalid_metrics = [
        metric for metric in METRICS
        if metric != 'snr_db' and not np.isfinite(frame[metric]).all()
    ]
    if invalid_metrics:
        raise ValueError(f'[{dataset_name}] metricas invalidas: {invalid_metrics}')
    return frame


def run_line(line_name, target_fs):
    frames = []
    for dataset_name, meta in DATASETS_META.items():
        print(f'[{line_name}] {dataset_name}: {meta["fs"]} -> {target_fs} Hz')
        frame = run_trial_metrics(
            raw_datasets[dataset_name], dataset_name, meta['fs'], target_fs, line_name
        )
        trial_cols = ['Subject', 'Activity_Code', 'Trial']
        expected_trials = raw_datasets[dataset_name][trial_cols].drop_duplicates().shape[0]
        processed_trials = frame[trial_cols].drop_duplicates().shape[0]
        if processed_trials != expected_trials:
            raise ValueError(
                f'[{dataset_name}] trials procesados {processed_trials} != esperados {expected_trials}'
            )
        frames.append(frame)
        print(f'  metricas: {len(frame):,} filas | trials: {processed_trials:,}')
    return pd.concat(frames, ignore_index=True)


In [ ]:
# Autoverificaciones basicas antes de procesar todos los trials.
assert get_poly_factors(100, 50) == (1, 2)
assert get_poly_factors(20, 50) == (5, 2)
assert get_poly_factors(100, 20) == (1, 5)
toy = np.arange(20, dtype=float)
assert np.array_equal(resample_signal(toy, 20, 20), toy)
assert len(resample_signal(toy, 20, 50)) == 50
assert len(resample_signal(toy, 100, 20)) == 4
print('Autoverificaciones de resampleo: OK')

## Funciones de resumen y visualizacion

In [ ]:
def finite_stats(values):
    values = np.asarray(values, dtype=float)
    finite = values[np.isfinite(values)]
    if len(finite) == 0:
        return np.inf, np.inf, np.inf
    return float(np.mean(finite)), float(np.median(finite)), float(np.quantile(finite, 0.95))


def summarize_metrics(metrics_df):
    rows = []
    group_cols = ['line', 'Dataset', 'Activity_Label', 'sensor']
    for keys, group in metrics_df.groupby(group_cols, sort=True):
        row = dict(zip(group_cols, keys))
        row['n_trials'] = int(group['Trial'].nunique())
        row['quality_pass_pct'] = float(group['quality_pass'].mean() * 100.0)
        row['interpolated_fraction_pct'] = float(group['interpolated_fraction'].iloc[0] * 100.0)
        for metric in METRICS:
            mean, median, p95 = finite_stats(group[metric])
            row[f'{metric}_mean'] = mean
            row[f'{metric}_median'] = median
            row[f'{metric}_p95'] = p95
        row['snr_infinite_pct'] = float(np.isinf(group['snr_db']).mean() * 100.0)
        rows.append(row)
    return pd.DataFrame(rows)


def select_representatives(metrics_df, line_name):
    avm = metrics_df[(metrics_df['line'] == line_name) & (metrics_df['sensor'] == 'AVM')].copy()
    selected = []
    group_cols = ['Dataset', 'Activity_Label']
    tie_cols = ['_distance', 'Subject', 'Activity_Code', 'Trial']
    for keys, group in avm.groupby(group_cols, sort=True):
        group = group.dropna(subset=['peak_atten_pct']).sort_values(
            ['Subject', 'Activity_Code', 'Trial'], kind='stable'
        )
        for quantile, label in QUANTILES:
            target_value = group['peak_atten_pct'].quantile(quantile)
            candidates = group.assign(_distance=(group['peak_atten_pct'] - target_value).abs())
            candidate = candidates.sort_values(tie_cols, kind='stable').iloc[0].to_dict()
            candidate['quantile'] = label
            candidate['quantile_value'] = float(target_value)
            selected.append(candidate)
    return pd.DataFrame(selected)


def get_trial(raw_df, row):
    mask = (
        (raw_df['Subject'] == row['Subject'])
        & (raw_df['Activity_Code'] == row['Activity_Code'])
        & (raw_df['Trial'] == row['Trial'])
    )
    return raw_df[mask].sort_values('Sample_Index')


def format_metric(value, decimals=2):
    return 'inf' if np.isinf(value) else f'{value:.{decimals}f}'


def show_inline(figure):
    display(figure)
    plt.close(figure)


def plot_metric_distributions(metrics_df, line_name):
    metric_titles = {
        'snr_db': 'SNR (dB) - mayor es mejor',
        'pearson_r': 'Pearson - mayor es mejor',
        'dtw_norm': 'DTW normalizado - menor es mejor',
        'phase_shift_ms': 'Desfase (ms) - menor es mejor',
        'peak_atten_pct': 'Error de pico (%) - menor es mejor',
    }
    groups = [(dataset, label) for dataset in DATASETS_META for label in sorted(VALID_LABELS)]
    labels = [f'{dataset}\n{label}' for dataset, label in groups]
    fig, axes = plt.subplots(2, len(METRICS), figsize=(22, 8), squeeze=False)
    fig.suptitle(f'{line_name}: distribucion de metricas por trial', fontweight='bold')
    for row_idx, sensor in enumerate(SENSORS):
        for col_idx, metric in enumerate(METRICS):
            ax = axes[row_idx, col_idx]
            values = []
            for dataset, label in groups:
                subset = metrics_df[
                    (metrics_df['line'] == line_name)
                    & (metrics_df['Dataset'] == dataset)
                    & (metrics_df['Activity_Label'] == label)
                    & (metrics_df['sensor'] == sensor)
                ][metric].to_numpy(dtype=float)
                values.append(subset[np.isfinite(subset)])
            valid_values = [value for value in values if len(value)]
            valid_positions = [idx + 1 for idx, value in enumerate(values) if len(value)]
            if valid_values:
                ax.boxplot(valid_values, positions=valid_positions, showfliers=False, patch_artist=True,
                           boxprops={'facecolor': '#d9e7f5'},
                           medianprops={'color': '#b22222', 'linewidth': 1.5})
                ax.set_xticks(range(1, len(labels) + 1))
                ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=7)
            else:
                ax.text(0.5, 0.5, 'Sin valores finitos\n(control identico)',
                        ha='center', va='center', transform=ax.transAxes)
            if metric == 'pearson_r':
                ax.axhline(THR_PEARSON_MIN, color='black', linestyle=':', linewidth=1)
            elif metric == 'phase_shift_ms':
                ax.axhline(THR_PHASE_MS_MAX, color='black', linestyle=':', linewidth=1)
            elif metric == 'peak_atten_pct':
                ax.axhline(THR_ATTEN_PCT_MAX, color='black', linestyle=':', linewidth=1)
            ax.set_title(f'{sensor}: {metric_titles[metric]}', fontsize=9)
            ax.grid(axis='y', alpha=0.25)
    plt.tight_layout()
    return fig


def plot_quality_rates(metrics_df, line_name):
    groups = [(dataset, label) for dataset in DATASETS_META for label in sorted(VALID_LABELS)]
    labels = [f'{dataset}\n{label}' for dataset, label in groups]
    fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
    fig.suptitle(f'{line_name}: porcentaje de trials que cumple todos los umbrales', fontweight='bold')
    for ax, sensor in zip(axes, SENSORS):
        rates = []
        for dataset, label in groups:
            subset = metrics_df[
                (metrics_df['line'] == line_name)
                & (metrics_df['Dataset'] == dataset)
                & (metrics_df['Activity_Label'] == label)
                & (metrics_df['sensor'] == sensor)
            ]
            rates.append(subset['quality_pass'].mean() * 100.0)
        bars = ax.bar(np.arange(len(labels)), rates, color='#5b8db8')
        ax.set_xticks(np.arange(len(labels)))
        ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
        ax.set_ylim(0, 105)
        ax.set_ylabel('Trials que cumplen (%)')
        ax.set_title(sensor)
        for bar, rate in zip(bars, rates):
            ax.text(bar.get_x() + bar.get_width() / 2, rate + 2, f'{rate:.1f}%',
                    ha='center', va='bottom', fontsize=8)
    plt.tight_layout()
    return fig

In [ ]:
def plot_time_grid(metrics_df, line_name, dataset_name, label, target_fs):
    reps = select_representatives(metrics_df, line_name)
    reps = reps[(reps['Dataset'] == dataset_name) & (reps['Activity_Label'] == label)]
    raw_df = raw_datasets[dataset_name]
    source_fs = DATASETS_META[dataset_name]['fs']
    fig, axes = plt.subplots(2, 3, figsize=(17, 7), squeeze=False, sharex=False)
    fig.suptitle(f'{line_name} | {dataset_name} | {label} | tiempo', fontweight='bold')
    for col_idx, (_, rep) in enumerate(reps.sort_values('quantile').iterrows()):
        trial = get_trial(raw_df, rep)
        analysis = analyze_trial(trial, source_fs, target_fs)
        for row_idx, sensor in enumerate(SENSORS):
            ax = axes[row_idx, col_idx]
            window = analysis['windows'][sensor]
            ax.plot(window['orig_times'], window['raw'], color='#777777', linewidth=1.0,
                    marker='o' if source_fs <= 20 else None, markersize=3, alpha=0.8,
                    label=f'Original {source_fs} Hz')
            if not np.allclose(window['raw'], window['reference']):
                ax.plot(window['orig_times'], window['reference'], color='#356d9b',
                        linestyle='--', linewidth=1.1, label='Referencia filtrada')
            ax.plot(window['res_times'], window['resampled'], color='#c94141' if target_fs == 50 else '#3a9d5d',
                    linewidth=1.5, marker='.' if target_fs > source_fs else None,
                    markersize=4, label=f'Resampleada {target_fs} Hz')
            metric_row = metrics_df[
                (metrics_df['line'] == line_name)
                & (metrics_df['Dataset'] == dataset_name)
                & (metrics_df['Activity_Label'] == label)
                & (metrics_df['Subject'] == rep['Subject'])
                & (metrics_df['Activity_Code'] == rep['Activity_Code'])
                & (metrics_df['Trial'] == rep['Trial'])
                & (metrics_df['sensor'] == sensor)
            ].iloc[0]
            ax.set_title(f'{sensor} - {rep["quantile"]} | pico {metric_row["peak_atten_pct"]:.2f}% | r {metric_row["pearson_r"]:.3f}', fontsize=9)
            ax.set_xlabel('Tiempo relativo (s)')
            ax.set_ylabel('Magnitud')
            ax.legend(fontsize=7)
    plt.tight_layout()
    return fig


def plot_psd_grid(metrics_df, line_name, dataset_name, label, target_fs):
    reps = select_representatives(metrics_df, line_name)
    reps = reps[(reps['Dataset'] == dataset_name) & (reps['Activity_Label'] == label)]
    raw_df = raw_datasets[dataset_name]
    source_fs = DATASETS_META[dataset_name]['fs']
    fig, axes = plt.subplots(2, 3, figsize=(17, 7), squeeze=False, sharex=False)
    fig.suptitle(f'{line_name} | {dataset_name} | {label} | PSD', fontweight='bold')
    for col_idx, (_, rep) in enumerate(reps.sort_values('quantile').iterrows()):
        trial = get_trial(raw_df, rep)
        analysis = analyze_trial(trial, source_fs, target_fs)
        for row_idx, sensor in enumerate(SENSORS):
            ax = axes[row_idx, col_idx]
            window = analysis['windows'][sensor]
            signals = [(window['raw'], source_fs, 'Original', '#777777')]
            if not np.allclose(window['raw'], window['reference']):
                signals.append((window['reference'], source_fs, 'Referencia filtrada', '#356d9b'))
            signals.append((window['resampled'], target_fs, f'Resampleada {target_fs} Hz', '#c94141' if target_fs == 50 else '#3a9d5d'))
            for signal, fs, name, color in signals:
                nperseg = min(128, len(signal))
                if nperseg < 4:
                    continue
                frequencies, power = welch(signal, fs=fs, nperseg=nperseg)
                ax.semilogy(frequencies, np.maximum(power, 1e-12), color=color, label=name)
            ax.axvline(source_fs / 2, color='#555555', linestyle=':', linewidth=1, label=f'Nyquist origen {source_fs / 2:g} Hz')
            if target_fs != source_fs:
                ax.axvline(target_fs / 2, color='black', linestyle='--', linewidth=1, label=f'Nyquist objetivo {target_fs / 2:g} Hz')
            ax.set_xlim(0, max(source_fs, target_fs) / 2 + 2)
            ax.set_title(f'{sensor} - {rep["quantile"]}', fontsize=9)
            ax.set_xlabel('Frecuencia (Hz)')
            ax.set_ylabel('PSD')
            ax.legend(fontsize=6)
    plt.tight_layout()
    return fig

# Linea 1: unificacion a 50 Hz

UPFall y KFall reducen de 100 Hz a 50 Hz. UMAFall aumenta de 20 Hz a 50 Hz; las muestras adicionales son interpoladas y no representan nueva observacion fisica.

In [ ]:
LINE_50 = 'Linea 1 - 50 Hz'
metrics_50 = run_line(LINE_50, 50)
summary_50 = summarize_metrics(metrics_50)

factor_rows = []
for dataset_name, meta in DATASETS_META.items():
    up, down = get_poly_factors(meta['fs'], 50)
    factor_rows.append({
        'Dataset': dataset_name,
        'Origen (Hz)': meta['fs'],
        'Objetivo (Hz)': 50,
        'up': up,
        'down': down,
        'Fraccion interpolada (%)': max(0.0, 1.0 - meta['fs'] / 50) * 100.0,
    })
display(pd.DataFrame(factor_rows))
display(summary_50)
show_inline(plot_metric_distributions(metrics_50, LINE_50))
show_inline(plot_quality_rates(metrics_50, LINE_50))

In [ ]:
for dataset_name in DATASETS_META:
    for label in sorted(VALID_LABELS):
        show_inline(plot_time_grid(metrics_50, LINE_50, dataset_name, label, 50))
        show_inline(plot_psd_grid(metrics_50, LINE_50, dataset_name, label, 50))

# Linea 2: unificacion a 20 Hz

UPFall y KFall reducen de 100 Hz a 20 Hz. UMAFall permanece en su frecuencia nativa de 20 Hz y funciona como control sin transformacion.

En el control nativo, `SNR=inf` significa error numerico cero. Se informa mediante `snr_infinite_pct` y no se interpreta como metricas faltantes.

In [ ]:
LINE_20 = 'Linea 2 - 20 Hz'
metrics_20 = run_line(LINE_20, 20)
summary_20 = summarize_metrics(metrics_20)

factor_rows = []
for dataset_name, meta in DATASETS_META.items():
    up, down = get_poly_factors(meta['fs'], 20)
    factor_rows.append({
        'Dataset': dataset_name,
        'Origen (Hz)': meta['fs'],
        'Objetivo (Hz)': 20,
        'up': up,
        'down': down,
        'Fraccion interpolada (%)': max(0.0, 1.0 - meta['fs'] / 20) * 100.0,
    })
display(pd.DataFrame(factor_rows))
display(summary_20)
show_inline(plot_metric_distributions(metrics_20, LINE_20))
show_inline(plot_quality_rates(metrics_20, LINE_20))

In [ ]:
for dataset_name in DATASETS_META:
    for label in sorted(VALID_LABELS):
        show_inline(plot_time_grid(metrics_20, LINE_20, dataset_name, label, 20))
        show_inline(plot_psd_grid(metrics_20, LINE_20, dataset_name, label, 20))

# Comparacion de las dos lineas

La comparacion usa las mismas unidades, etiquetas y trials. Las figuras de distribucion muestran como cambia cada metrica al cambiar la frecuencia objetivo. Las figuras de superposicion reutilizan un mismo trial para aislar el efecto de la frecuencia.

Las secciones individuales seleccionan P05/P50/P95 por linea. La comparacion directa selecciona trials comunes mediante el promedio del error de pico AVM de ambas lineas, para que 50 Hz y 20 Hz se comparen sobre el mismo evento.

In [ ]:
metrics_all = pd.concat([metrics_50, metrics_20], ignore_index=True)
summary_all = summarize_metrics(metrics_all)
comparison_keys = ['Dataset', 'Activity_Label', 'sensor']
comparison_table = (
    summary_all[summary_all['line'] == LINE_50]
    .merge(
        summary_all[summary_all['line'] == LINE_20],
        on=comparison_keys,
        suffixes=('_50Hz', '_20Hz'),
    )
)
comparison_table['delta_quality_pass_pct_20_minus_50'] = (
    comparison_table['quality_pass_pct_20Hz']
    - comparison_table['quality_pass_pct_50Hz']
)
comparison_table['delta_peak_p95_20_minus_50'] = (
    comparison_table['peak_atten_pct_p95_20Hz']
    - comparison_table['peak_atten_pct_p95_50Hz']
)
display(comparison_table)

In [ ]:
def plot_comparison_distributions(metrics_50, metrics_20):
    groups = [(dataset, label) for dataset in DATASETS_META for label in sorted(VALID_LABELS)]
    labels = [f'{dataset}\n{label}' for dataset, label in groups]
    fig, axes = plt.subplots(2, len(METRICS), figsize=(22, 8), squeeze=False)
    fig.suptitle('Comparacion de distribuciones: 50 Hz frente a 20 Hz', fontweight='bold')
    for row_idx, sensor in enumerate(SENSORS):
        for col_idx, metric in enumerate(METRICS):
            ax = axes[row_idx, col_idx]
            positions = []
            values = []
            tick_positions = []
            for idx, (dataset, label) in enumerate(groups):
                for offset, frame in [(-0.18, metrics_50), (0.18, metrics_20)]:
                    subset = frame[
                        (frame['Dataset'] == dataset)
                        & (frame['Activity_Label'] == label)
                        & (frame['sensor'] == sensor)
                    ][metric].to_numpy(dtype=float)
                    values.append(subset[np.isfinite(subset)])
                    positions.append(idx + 1 + offset)
                tick_positions.append(idx + 1)
            for value, position, color in zip(values, positions, ['#d66a6a', '#62a778'] * len(groups)):
                if len(value):
                    box = ax.boxplot([value], positions=[position], widths=0.28,
                                    showfliers=False, patch_artist=True)
                    box['boxes'][0].set_facecolor(color)
                    box['medians'][0].set_color('black')
            ax.set_xticks(tick_positions)
            ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=7)
            ax.set_title(f'{sensor}: {metric}', fontsize=9)
            if metric == 'pearson_r':
                ax.axhline(THR_PEARSON_MIN, color='black', linestyle=':', linewidth=1)
            elif metric == 'phase_shift_ms':
                ax.axhline(THR_PHASE_MS_MAX, color='black', linestyle=':', linewidth=1)
            elif metric == 'peak_atten_pct':
                ax.axhline(THR_ATTEN_PCT_MAX, color='black', linestyle=':', linewidth=1)
    fig.text(0.91, 0.96, 'rojo: 50 Hz | verde: 20 Hz', ha='right', fontsize=9)
    plt.tight_layout()
    return fig


def plot_quality_heatmap(metrics_50, metrics_20):
    groups = [(dataset, label) for dataset in DATASETS_META for label in sorted(VALID_LABELS)]
    labels = [f'{dataset}\n{label}' for dataset, label in groups]
    matrix = []
    for dataset, label in groups:
        rates = []
        for frame in (metrics_50, metrics_20):
            subset = frame[
                (frame['Dataset'] == dataset)
                & (frame['Activity_Label'] == label)
                & (frame['sensor'] == 'AVM')
            ]
            rates.append(subset['quality_pass'].mean() * 100.0)
        matrix.append(rates)
    matrix = np.asarray(matrix, dtype=float)
    fig, ax = plt.subplots(figsize=(6, 6))
    image = ax.imshow(matrix, cmap='YlGnBu', vmin=0, vmax=100, aspect='auto')
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['50 Hz', '20 Hz'])
    ax.set_yticks(np.arange(len(labels)))
    ax.set_yticklabels(labels)
    for row_idx in range(matrix.shape[0]):
        for col_idx in range(matrix.shape[1]):
            ax.text(col_idx, row_idx, f'{matrix[row_idx, col_idx]:.1f}%',
                    ha='center', va='center', fontsize=9)
    ax.set_title('AVM: cumplimiento de umbrales por linea')
    fig.colorbar(image, ax=ax, label='Trials que cumplen (%)')
    plt.tight_layout()
    return fig


def select_comparison_representatives(metrics_50, metrics_20):
    keys = ['Dataset', 'Activity_Label', 'Subject', 'Activity_Code', 'Trial']
    left = metrics_50[metrics_50['sensor'] == 'AVM'][keys + ['peak_atten_pct']].rename(columns={'peak_atten_pct': 'peak_50'})
    right = metrics_20[metrics_20['sensor'] == 'AVM'][keys + ['peak_atten_pct']].rename(columns={'peak_atten_pct': 'peak_20'})
    merged = left.merge(right, on=keys, how='inner')
    merged['combined_peak_error'] = (merged['peak_50'] + merged['peak_20']) / 2.0
    selected = []
    for group_keys, group in merged.groupby(['Dataset', 'Activity_Label'], sort=True):
        group = group.sort_values(['Subject', 'Activity_Code', 'Trial'], kind='stable')
        for quantile, label in QUANTILES:
            target_value = group['combined_peak_error'].quantile(quantile)
            candidate = (
                group.assign(_distance=(group['combined_peak_error'] - target_value).abs())
                .sort_values(['_distance', 'Subject', 'Activity_Code', 'Trial'], kind='stable')
                .iloc[0]
                .to_dict()
            )
            candidate['quantile'] = label
            selected.append(candidate)
    return pd.DataFrame(selected)


show_inline(plot_comparison_distributions(metrics_50, metrics_20))
show_inline(plot_quality_heatmap(metrics_50, metrics_20))

In [ ]:
def plot_direct_comparison(dataset_name, label):
    reps = select_comparison_representatives(metrics_50, metrics_20)
    reps = reps[(reps['Dataset'] == dataset_name) & (reps['Activity_Label'] == label)]
    raw_df = raw_datasets[dataset_name]
    source_fs = DATASETS_META[dataset_name]['fs']
    fig, axes = plt.subplots(2, 3, figsize=(17, 7), squeeze=False)
    fig.suptitle(f'Comparacion directa | {dataset_name} | {label}', fontweight='bold')
    for col_idx, (_, rep) in enumerate(reps.sort_values('quantile').iterrows()):
        trial = get_trial(raw_df, rep)
        analysis_50 = analyze_trial(trial, source_fs, 50)
        analysis_20 = analyze_trial(trial, source_fs, 20)
        for row_idx, sensor in enumerate(SENSORS):
            ax = axes[row_idx, col_idx]
            window = analysis_50['windows'][sensor]
            ax.plot(window['orig_times'], window['raw'], color='#777777', linewidth=1.0,
                    marker='o' if source_fs <= 20 else None, markersize=3, label=f'Original {source_fs} Hz')
            ax.plot(analysis_50['windows'][sensor]['res_times'], analysis_50['windows'][sensor]['resampled'],
                    color='#c94141', linewidth=1.5, label='50 Hz')
            ax.plot(analysis_20['windows'][sensor]['res_times'], analysis_20['windows'][sensor]['resampled'],
                    color='#3a9d5d', linewidth=1.5, label='20 Hz')
            ax.set_title(f'{sensor} - {rep["quantile"]} | errores AVM: 50={rep["peak_50"]:.2f}% / 20={rep["peak_20"]:.2f}%', fontsize=9)
            ax.set_xlabel('Tiempo relativo (s)')
            ax.set_ylabel('Magnitud')
            ax.legend(fontsize=7)
    plt.tight_layout()
    return fig


for dataset_name in DATASETS_META:
    for label in sorted(VALID_LABELS):
        show_inline(plot_direct_comparison(dataset_name, label))

In [ ]:
def plot_direct_psd_comparison(dataset_name, label):
    reps = select_comparison_representatives(metrics_50, metrics_20)
    reps = reps[(reps['Dataset'] == dataset_name) & (reps['Activity_Label'] == label)]
    raw_df = raw_datasets[dataset_name]
    source_fs = DATASETS_META[dataset_name]['fs']
    fig, axes = plt.subplots(2, 3, figsize=(17, 7), squeeze=False)
    fig.suptitle(f'Comparacion directa | {dataset_name} | {label} | PSD', fontweight='bold')
    for col_idx, (_, rep) in enumerate(reps.sort_values('quantile').iterrows()):
        trial = get_trial(raw_df, rep)
        analysis_50 = analyze_trial(trial, source_fs, 50)
        analysis_20 = analyze_trial(trial, source_fs, 20)
        for row_idx, sensor in enumerate(SENSORS):
            ax = axes[row_idx, col_idx]
            window = analysis_50['windows'][sensor]
            signals = [
                (window['raw'], source_fs, 'Original', '#777777'),
                (analysis_50['windows'][sensor]['resampled'], 50, '50 Hz', '#c94141'),
                (analysis_20['windows'][sensor]['resampled'], 20, '20 Hz', '#3a9d5d'),
            ]
            for signal, fs, name, color in signals:
                nperseg = min(128, len(signal))
                if nperseg < 4:
                    continue
                frequencies, power = welch(signal, fs=fs, nperseg=nperseg)
                ax.semilogy(frequencies, np.maximum(power, 1e-12), color=color, label=name)
            ax.axvline(source_fs / 2, color='#555555', linestyle=':', linewidth=1, label=f'Nyquist origen {source_fs / 2:g} Hz')
            ax.axvline(25, color='#c94141', linestyle='--', linewidth=1, label='Nyquist 50 Hz')
            ax.axvline(10, color='#3a9d5d', linestyle='--', linewidth=1, label='Nyquist 20 Hz')
            ax.set_xlim(0, max(source_fs, 50) / 2 + 2)
            ax.set_title(f'{sensor} - {rep["quantile"]}', fontsize=9)
            ax.set_xlabel('Frecuencia (Hz)')
            ax.set_ylabel('PSD')
            ax.legend(fontsize=6)
    plt.tight_layout()
    return fig


for dataset_name in DATASETS_META:
    for label in sorted(VALID_LABELS):
        show_inline(plot_direct_psd_comparison(dataset_name, label))

In [ ]:
def build_decision_summary(metrics_50, metrics_20):
    rows = []
    for dataset_name in DATASETS_META:
        for label in sorted(VALID_LABELS):
            candidates = {}
            for target, frame in [(50, metrics_50), (20, metrics_20)]:
                avm = frame[
                    (frame['Dataset'] == dataset_name)
                    & (frame['Activity_Label'] == label)
                    & (frame['sensor'] == 'AVM')
                ]
                finite_pearson = avm['pearson_r'].replace([np.inf, -np.inf], np.nan).dropna()
                candidates[target] = {
                    'pass_pct': avm['quality_pass'].mean() * 100.0,
                    'peak_p95': finite_stats(avm['peak_atten_pct'])[2],
                    'pearson_p05': float(finite_pearson.quantile(0.05)) if len(finite_pearson) else np.nan,
                }
            key_50 = (-candidates[50]['pass_pct'], candidates[50]['peak_p95'], -candidates[50]['pearson_p05'])
            key_20 = (-candidates[20]['pass_pct'], candidates[20]['peak_p95'], -candidates[20]['pearson_p05'])
            recommended = 50 if key_50 < key_20 else 20
            rows.append({
                'Dataset': dataset_name,
                'Activity_Label': label,
                'Recomendacion segun AVM': f'{recommended} Hz',
                'Cumplimiento 50 Hz (%)': candidates[50]['pass_pct'],
                'Cumplimiento 20 Hz (%)': candidates[20]['pass_pct'],
                'P95 error pico 50 Hz (%)': candidates[50]['peak_p95'],
                'P95 error pico 20 Hz (%)': candidates[20]['peak_p95'],
                'P05 Pearson 50 Hz': candidates[50]['pearson_p05'],
                'P05 Pearson 20 Hz': candidates[20]['pearson_p05'],
            })
    return pd.DataFrame(rows)

decision_summary = build_decision_summary(metrics_50, metrics_20)
display(decision_summary)
print('Criterio de recomendacion: maximizar cumplimiento AVM; empate -> menor P95 de error de pico; nuevo empate -> mayor P05 de Pearson.')

## Lectura final

Interpretar siempre junto con las figuras temporales y PSD:

- Una media baja de error no descarta degradacion en la cola P95.
- `20 -> 50 Hz` puede suavizar o crear sobreimpulsos sin aportar informacion fisica por encima de `10 Hz`.
- `100 -> 20 Hz` elimina de forma irreversible contenido por encima de `10 Hz`; la conveniencia depende de si ese contenido es relevante para detectar caidas.
- La tabla de decision es una ayuda cuantitativa, no reemplaza la inspeccion visual de impactos y espectros.